Make sure the right catalog and schema are used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

# Create silver tables
## The customer table

In [0]:
CREATE TABLE IF NOT EXISTS customer (
        customer_id STRING NOT NULL,
        last_name STRING,
        first_name STRING,
        signup_date DATE,
        plan_tier STRING,
        plz STRING,
        street STRING,
        city STRING,
        zone STRING,
        contract_type STRING,
        autopay_enabled BOOLEAN,
        payment_method STRING,
        monthly_bill DOUBLE,
        speed_tier_mbps INT,
        data_usage_gb_last_month DOUBLE,
        CONSTRAINT customer_pk PRIMARY KEY (customer_id)
    );

ALTER TABLE
    customer
DROP CONSTRAINT IF EXISTS
    customer_sd;

ALTER TABLE
    customer
DROP CONSTRAINT IF EXISTS
    customer_mb;

ALTER TABLE
    customer
DROP CONSTRAINT IF EXISTS
    customer_du;

ALTER TABLE
    customer
ADD
    CONSTRAINT customer_sd CHECK (signup_date <= CURRENT_DATE());

ALTER TABLE
    customer
ADD
    CONSTRAINT customer_mb CHECK (monthly_bill >= 0);

ALTER TABLE
    customer
ADD
    CONSTRAINT customer_du CHECK (data_usage_gb_last_month >= 0 OR data_usage_gb_last_month IS NULL);

## The churn prediction table

In [0]:
CREATE TABLE IF NOT EXISTS churn (
        customer_id STRING NOT NULL,
        churned BOOLEAN,
        churn_reason STRING,
        CONSTRAINT churn_pk PRIMARY KEY (customer_id),
        CONSTRAINT churn_fk FOREIGN KEY (customer_id) REFERENCES customer (customer_id)
    );

## The connection log table

In [0]:
CREATE TABLE IF NOT EXISTS log (
        timestamp TIMESTAMP,
        customer_id STRING,
        issue_detected STRING,
        speed_measured_mbps DOUBLE,
        packet_loss_percent DOUBLE,
        latency_ms DOUBLE,
        downtime_minutes INTEGER,
        connection_drops_count INTEGER,
        CONSTRAINT log_pk PRIMARY KEY (timestamp),
        CONSTRAINT log_fk FOREIGN KEY (customer_id) REFERENCES customer (customer_id)
    );

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_ts;

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_sm;

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_pl;

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_l;

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_dt;

ALTER TABLE
    log
DROP CONSTRAINT IF EXISTS
    log_connection_cd;

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_ts CHECK (timestamp <= CURRENT_DATE());

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_sm CHECK (speed_measured_mbps >= 0);

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_pl
        CHECK (
            packet_loss_percent >= 0
            AND packet_loss_percent <= 100
        );

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_l CHECK (latency_ms >= 0);

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_dt CHECK (downtime_minutes >= 0);

ALTER TABLE
    log
ADD
    CONSTRAINT log_connection_cd CHECK (connection_drops_count >= 0);

## The support ticket table

In [0]:
CREATE TABLE IF NOT EXISTS ticket (
        ticket_id STRING NOT NULL,
        customer_id STRING,
        timestamp_created TIMESTAMP,
        timestamp_closed TIMESTAMP,
        subject STRING,
        description STRING,
        priority STRING,
        status STRING,
        CONSTRAINT ticket_pk PRIMARY KEY (ticket_id),
        CONSTRAINT ticket_f1 FOREIGN KEY (customer_id) REFERENCES customer (customer_id)
    );

ALTER TABLE
    ticket
DROP CONSTRAINT IF EXISTS
    support_ticket_ci;

ALTER TABLE
    ticket
DROP CONSTRAINT IF EXISTS
    support_ticket_tc;

ALTER TABLE
    ticket
DROP CONSTRAINT IF EXISTS
    support_ticket_tl;

ALTER TABLE
    ticket
ADD
    CONSTRAINT support_ticket_ci CHECK (customer_id IS NOT NULL);

ALTER TABLE
    ticket
ADD
    CONSTRAINT support_ticket_tc
        CHECK (
            (
                status = 'open'
                AND timestamp_closed IS NULL
            )
            OR (
                status != 'open'
                AND timestamp_created <= timestamp_closed
            )
        );

ALTER TABLE
    ticket
ADD
    CONSTRAINT support_ticket_tl
        CHECK (
            timestamp_created <= CURRENT_DATE()
            OR (
                timestamp_closed <= CURRENT_DATE()
                AND status != 'open'
            )
        );

## The agent table

In [0]:
CREATE TABLE IF NOT EXISTS agent (
        agent_id STRING NOT NULL,
        first_name STRING,
        last_name STRING,
        employment_date DATE,
        experience_level STRING,
        monthly_salary_eur INTEGER,
        CONSTRAINT agent_pk PRIMARY KEY (agent_id)
    );

ALTER TABLE
    agent
DROP CONSTRAINT IF EXISTS
    agent_ed;

ALTER TABLE
    agent
DROP CONSTRAINT IF EXISTS
    agent_ms;

ALTER TABLE
    agent
ADD
    CONSTRAINT agent_ed CHECK (employment_date <= CURRENT_DATE());

ALTER TABLE
    agent
ADD
    CONSTRAINT agent_ms CHECK (monthly_salary_eur >= 0);

## The chat table

In [0]:
CREATE TABLE IF NOT EXISTS chat (
        session_id STRING NOT NULL,
        customer_id STRING,
        agent_id STRING,
        classification STRING,
        comment STRING,
        CONSTRAINT chat_pk PRIMARY KEY (session_id),
        CONSTRAINT chat_f1 FOREIGN KEY (customer_id) REFERENCES customer (customer_id),
        CONSTRAINT chat_f2 FOREIGN KEY (agent_id) REFERENCES agent (agent_id)
    );

ALTER TABLE
    chat
DROP CONSTRAINT IF EXISTS
    chat_transcript_ci;

ALTER TABLE
    chat
DROP CONSTRAINT IF EXISTS
    chat_transcript_ai;
    
ALTER TABLE
    chat
ADD
    CONSTRAINT chat_transcript_ci CHECK (customer_id IS NOT NULL);

ALTER TABLE
    chat
ADD
    CONSTRAINT chat_transcript_ai CHECK (agent_id IS NOT NULL);

## The message table

In [0]:
CREATE TABLE IF NOT EXISTS message (
        session_id STRING NOT NULL,
        speaker STRING,
        timestamp TIMESTAMP NOT NULL,
        message STRING,
        sentiment STRING,
        CONSTRAINT message_fk FOREIGN KEY (session_id) REFERENCES chat (session_id),
        CONSTRAINT message_pk PRIMARY KEY (session_id, speaker, timestamp)
    );

ALTER TABLE
    message
DROP CONSTRAINT IF EXISTS
    message_ts;

ALTER TABLE
    message
ADD
    CONSTRAINT message_ts
        CHECK (
            timestamp IS NOT NULL
            AND timestamp <= CURRENT_TIMESTAMP()
        );

# Fill tables with content
## Customer table

In [0]:
WITH f_loc AS (
	SELECT
		customer_id,
		ingestion_time,
		substring_index(address, ',', 1) AS street,
		substr(address, len(street) + 2) AS rest
	FROM
		customer_bronze
),
location AS (
	SELECT
		customer_id,
		ingestion_time,
		CASE
			WHEN
				LEN(regexp_extract(rest, '([0-9]+)', 1)) = 4
			THEN
				CONCAT('0', regexp_extract(rest, '([0-9]+)', 1))
			ELSE regexp_extract(rest, '([0-9]+)', 1)
		END AS plz,
		replace(street, 'str.', 'straße') AS street,
		CASE
			WHEN rest LIKE '%berlin%' THEN 'Berlin'
			WHEN rest LIKE '%None%' THEN trim(replace(rest, 'None ', ''))
			ELSE substring_index(trim(regexp_extract(rest, '\\d{4,5}\\s+(.+?)\\s*$', 1)), ',', 1)
		END AS city
	FROM
		f_loc
) MERGE INTO
	customer s
USING (
	SELECT
		c.customer_id,
		right(c.customer_name, len(c.customer_name) - charindex(' ', c.customer_name)) AS last_name,
		left(c.customer_name, charindex(' ', c.customer_name)) AS first_name,
		CAST(c.signup_date AS DATE) AS signup_date,
		c.plan_tier,
		l.plz AS plz,
		l.street AS street,
		replace(l.city, 'Stadt ', '') AS city,
		p.bundesland AS zone,
		c.contract_type,
		CASE
			WHEN LOWER(c.autopay_enabled) LIKE 'true' THEN TRUE
			ELSE FALSE
		END AS autopay_enabled,
		c.payment_method,
		CAST(c.monthly_bill AS DOUBLE) AS monthly_bill,
		CASE
			WHEN regexp_extract(c.plan_tier, '([0-9]+)G', 1) = '1' THEN int(1000)
			ELSE cast(regexp_extract(c.plan_tier, '([0-9]+)', 1) as int)
		END AS speed_tier_mbps,
		CAST(c.data_usage_gb_last_month AS DOUBLE) AS data_usage_gb_last_month
	FROM
		customer_bronze c
			JOIN location l
				ON c.customer_id = l.customer_id
				AND c.ingestion_time = l.ingestion_time
			LEFT JOIN plz_to_state p
				ON cast(p.plz as int)
					= try_cast(
						regexp_extract(substring_index(c.address, ',', -1), '([0-9]+)', 1) as int
					)
	QUALIFY
		row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
	b.customer_id = s.customer_id
WHEN MATCHED AND
	sha1(
		concat_ws(
			'|',
			b.last_name,
			b.first_name,
			b.signup_date,
			b.plan_tier,
			b.street,
			b.city,
			b.contract_type,
			b.autopay_enabled,
			b.payment_method,
			b.monthly_bill,
			b.speed_tier_mbps,
			b.data_usage_gb_last_month
		)
	)
		!= sha1(
			concat_ws(
				'|',
				s.last_name,
				s.first_name,
				s.signup_date,
				s.plan_tier,
				s.street,
				s.city,
				s.contract_type,
				s.autopay_enabled,
				s.payment_method,
				s.monthly_bill,
				s.speed_tier_mbps,
				s.data_usage_gb_last_month
			)
		)
	THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


Teilweise wurden keine PLZ angegeben → AI wird fehlende Werte basierend auf der Gemeinde und Straße ermitteln.
Von der AI zu analysierende Werte werden eingeschränkt um benötigte Zeit zu reduzieren

In [0]:
CREATE OR REPLACE VIEW ai_address AS
SELECT
	customer_id,
	street,
	city,
	ai_query(
		"databricks-gemma-3-12b",
		request =>
			concat(
				"Gebe basierend auf einem Gemeindenamen und einer Straße aus der Gemeinde als Ergebnis eine JSON Array aus,das die zugehörige Postleitzahl (PLZ) und das deutsche Bundesland enthält.

Beispiel:

street
Über der Mühle 3
city
Ilmtal-Weinstraße

RESULT
[
{'plz': '99510','bundeland': 'Thüringen'}
]

DOCUMENT\n",
				street,
				city,
				'\n\nRESULT\n'
			),
		responseFormat =>
			'{
              "type": "json_schema",
              "json_schema": {
                  "name": "address_schema",
                  "schema": {
                      "type": "array",
                      "items": {
                          "type": "object",
                          "properties": {
                              "plz": { "type": "string" },
                              "bundesland": { "type": "string" }
                          }
                      }
                  },
                  "strict": true
              }
          }'
	) as address
FROM
	customer
WHERE
	plz = '';

MERGE INTO
	customer c
USING (
	SELECT
		customer_id,
		ai.plz,
		ai.bundesland
	FROM
		ai_address
		LATERAL VIEW OUTER EXPLODE(FROM_JSON(ai_address.address, 'ARRAY<MAP<STRING,STRING>>')) AS ai
	QUALIFY
		ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ai.plz) = 1
) a
ON
	c.customer_id = a.customer_id
WHEN MATCHED THEN UPDATE SET c.plz = a.plz, c.zone = a.bundesland

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## Churn prediction table

In [0]:
MERGE INTO
    churn s
USING (
    SELECT
        c.customer_id,
        CASE
            WHEN c.churned = 1 THEN TRUE
            ELSE FALSE
        END AS churned,
        c.churn_reason
    FROM
        churn_bronze c
    QUALIFY
        row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    b.customer_id = s.customer_id
WHEN MATCHED AND sha1(concat_ws('|', b.churned, b.churn_reason)) != 
                 sha1(concat_ws('|', s.churned, s.churn_reason))  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## Connection log table

In [0]:
MERGE INTO
    log s
USING (
    SELECT
        CAST(l.timestamp AS timestamp) AS timestamp,
        l.customer_id,
        l.issue_detected,
        CAST(l.speed_measured_mbps AS double) AS speed_measured_mbps,
        CAST(l.packet_loss_percent AS double) AS packet_loss_percent,
        CAST(l.latency_ms AS double) AS latency_ms,
        CAST(l.downtime_minutes AS integer) AS downtime_minutes,
        CAST(l.connection_drops_count AS integer) AS connection_drops_count
    FROM
        log_bronze l
    QUALIFY
        row_number() OVER (PARTITION BY l.timestamp ORDER BY l.ingestion_time DESC) = 1
) b
ON
    s.timestamp = b.timestamp
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
340887,0,0,340887


## Support ticket table

In [0]:
MERGE INTO
    ticket s
USING (
    SELECT
        t.ticket_id,
        t.customer_id,
        CAST(t.timestamp_created AS timestamp) AS timestamp_created,
        CASE
            WHEN t.timestamp_closed = 'nan' THEN NULL
            ELSE CAST(t.timestamp_closed AS timestamp)
        END AS timestamp_closed,
        t.subject,
        t.description,
        t.priority,
        t.status
    FROM
        ticket_bronze t
    QUALIFY
        row_number() OVER (PARTITION BY t.ticket_id ORDER BY t.ingestion_time DESC) = 1
) b
ON
    s.ticket_id = b.ticket_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
500,0,0,500


## Agent table

In [0]:
MERGE INTO
    agent s
USING (
    SELECT
        a.agent_id,
        SUBSTRING(a.agent_name, 1, POSITION(' ' IN a.agent_name) - 1) AS first_name,
        SUBSTRING(a.agent_name, POSITION(' ' IN a.agent_name) + 1) AS last_name,
        TO_DATE(a.employment_date) AS employment_date,
        a.experience_level,
        CAST(a.monthly_salary_eur AS int) AS monthly_salary_eur
    FROM
        agent_bronze a
    QUALIFY
        row_number() OVER (PARTITION BY a.agent_id ORDER BY a.ingestion_time DESC) = 1
) b
ON
    s.agent_id = b.agent_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
20,0,0,20


## Temporary table for the AI Query

In [0]:
CREATE OR REPLACE VIEW ai_analyse_message AS
WITH messages_raw AS (
    SELECT
        c.session_id,
        msg.speaker,
        CAST(msg.timestamp AS timestamp) AS timestamp,
        msg.message,
        c.ingestion_time
    FROM
        chat_bronze c
        LEFT JOIN chat ch ON c.session_id = ch.session_id
        LATERAL VIEW EXPLODE(FROM_JSON(c.messages, 'ARRAY<MAP<STRING,STRING>>')) AS msg
    WHERE ch.session_id IS NULL
),
opinion AS (
    SELECT
        messages_raw.session_id,
        messages_raw.speaker,
        messages_raw.timestamp,
        ai_query(
            "databricks-gemma-3-12b",
            request =>
                concat(
                    "Du bist ein Internet Service Provider. Gebe basierend auf einem Textabschnitt als Ergebnis ein JSON Array aus, das eine Zusammenfassung, eine Klassifikation und ein Positiv, Negativ oder Neutral Sentiment über das Thema enthält. Klassifiziert muss einer der folgenden Anworten sein: CONNECTION ISSUES, SLOW SPEED, SERVICE, PRICE, OTHER. Du kannst keine Klassifikations Kategorie halluzinieren.

Beispiel:

DOCUMENT
Mein Router startet sich alle 15 Minuten von selbst neu. Das ist super nervig. (Gemessene Geschwindigkeit: 180 Mbps, Issue: packet_loss).

RESULT
[
{'Classification': 'CONNECTION ISSUES','Comment': 'Router startet ständig neu','Sentiment': 'Negativ'}
]

DOCUMENT\n",
                    messages_raw.message,
                    '\n\nRESULT\n'
                ),
            responseFormat =>
                '{
                "type": "json_schema",
                "json_schema": {
                    "name": "opinion_mining_schema",
                    "schema": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "classification": { "type": "string" },
                                "comment": { "type": "string" },
                                "sentiment": { "type": "string" }
                            }
                        }
                    },
                    "strict": true
                }
            }'
        ) as extracted_opinions
    FROM
        messages_raw
) 
SELECT
    m.session_id,
    m.speaker,
    m.timestamp,
    m.message,
    m.ingestion_time,
    op.classification,
    op.comment,
    op.sentiment
FROM
    messages_raw m
            LEFT JOIN opinion o
                ON m.session_id = o.session_id
                AND m.timestamp = o.timestamp
        LATERAL VIEW OUTER
            EXPLODE(FROM_JSON(o.extracted_opinions, 'ARRAY<MAP<STRING,STRING>>')) AS op

## Chat table

In [0]:
MERGE INTO
    chat s
USING (
    SELECT
        c.session_id,
        c.customer_id,
        c.agent_id,
        FIRST(m.classification) AS classification,
        FIRST(m.comment) AS comment
    FROM
        chat_bronze c
        LEFT JOIN ai_analyse_message m ON c.session_id = m.session_id
    GROUP BY
        c.session_id, c.customer_id, c.agent_id, c.ingestion_time
    QUALIFY
        row_number() OVER (PARTITION BY c.session_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
    s.session_id = b.session_id
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
495,0,0,495


message

In [0]:
MERGE INTO
    message s
USING (
    SELECT
        session_id,
        speaker,
        timestamp,
        message,
        classification,
        comment,
        CASE
            WHEN sentiment IN ('Positive', 'Positiv', 'Positives') THEN 'Positiv'
            WHEN sentiment IN ('Neutral', 'Neutrales') THEN 'Neutral'
            WHEN sentiment IN ('Negative', 'Negativ', 'Negatives') THEN 'Negativ'
            ELSE 'Unbekannt'
        END AS sentiment
    FROM
        ai_analyse_message
    QUALIFY
        row_number() OVER (PARTITION BY session_id, timestamp ORDER BY ingestion_time DESC) = 1
) b
ON
    b.session_id = s.session_id
    AND b.speaker = s.speaker
    AND b.timestamp = s.timestamp
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0
